In [1]:

import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

df = pd.read_csv("aligned_pairs_by_id.csv")
es = df["es_text"].fillna("").tolist()
en = df["en_text"].fillna("").tolist()

model = SentenceTransformer("sentence-transformers/LaBSE")
emb_es = model.encode(es, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb_en = model.encode(en, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

# coseno por pares (embeddings normalizados -> producto escalar fila a fila)
cos = np.sum(emb_es * emb_en, axis=1)
df["labse_cosine"] = cos

print(f"n pares        : {len(df)}")
print(f"media coseno   : {cos.mean():.4f}")
print(f"mediana coseno : {np.median(cos):.4f}")
print(f"desv. típica   : {cos.std():.4f}")
for thr in (0.7, 0.75, 0.8, 0.85):
    print(f"  >= {thr:.2f} : {100*np.mean(cos>=thr):.1f}%")

df.to_csv("aligned_pairs_with_labse.csv", index=False)
print("Guardado: aligned_pairs_with_labse.csv")

# Revisar manualmente la cola baja (posibles traducciones flojas o ruido):
low = df.nsmallest(30, "labse_cosine")[["id_str","labse_cosine","es_text","en_text"]]
low.to_csv("labse_lowest30.csv", index=False)
print("Guardado: labse_lowest30.csv (cola baja para inspección manual)")

Batches:   0%|          | 0/234 [00:00<?, ?it/s]

Batches:   0%|          | 0/234 [00:00<?, ?it/s]

n pares        : 14952
media coseno   : 0.9123
mediana coseno : 0.9264
desv. típica   : 0.0653
  >= 0.70 : 98.5%
  >= 0.75 : 98.1%
  >= 0.80 : 97.1%
  >= 0.85 : 92.3%
Guardado: aligned_pairs_with_labse.csv
Guardado: labse_lowest30.csv (cola baja para inspección manual)
